# 05.02 — Profile vs. Definition

This notebook re-exercises the original profile-vs-definition comparison
using the renamed API introduced in E27:

| Old (removed) | New |
|---|---|
| `compare(profile, graph_definition)` | `compare_profile_to_definition(profile, graph_definition)` |

The semantics and emitted codes are **identical** — only the function name
changed. `compare_profile_to_definition` still uses `standard_rules()` which
asks: *does the observed profile satisfy the declared definition?*

All three comparison functions live in `orthograph.comparison.engine`:

```python
from orthograph.comparison.engine import (
    compare_profile_to_definition,  # satisfaction check (ERROR/WARNING/INFO)
    compare_profiles,               # symmetric diff of two profiles (INFO only)
    compare_definitions,            # symmetric diff of two definitions (INFO only)
)
```

This notebook:
1. Builds a `GraphDefinition` and two `GraphProfile` objects (one valid, one with issues).
2. Calls `compare_profile_to_definition` and renders the `ValidationResult`.
3. Shows how `is_valid` and `errors` work for actionable feedback.

In [1]:
from orthograph.comparison.engine import compare_profile_to_definition
from orthograph.diagnostics.result import GraphValidationError
from orthograph.graph_profile.models import (
    CardinalityStats,
    GraphProfile,
    NodeTypeProfile,
    PropertyProfile,
    RelationshipTypeProfile,
)

## 1. Declare the graph model

In [2]:
from shared.filmography import FILMOGRAPHY_MODEL


graph_definition = FILMOGRAPHY_MODEL
print("Model:", graph_definition.name, "| nodes:", sorted(graph_definition.node_labels))

Model: Filmography | nodes: ['City', 'Movie', 'Person']


## 2. Build a profile that satisfies the definition

All required properties present, correct types, endpoints and cardinality within bounds.

In [3]:
valid_profile = GraphProfile(
    source="neo4j://prod:7687",
    node_type_profiles={
        "Person": NodeTypeProfile(
            label="Person",
            count=120,
            property_profiles={
                "name": PropertyProfile(
                    name="name",
                    present_count=120,
                    total_count=120,
                    observed_types=["String"],
                ),
                "born": PropertyProfile(
                    name="born",
                    present_count=90,
                    total_count=120,
                    observed_types=["Long"],
                ),
            },
        ),
        "Movie": NodeTypeProfile(
            label="Movie",
            count=38,
            property_profiles={
                "title": PropertyProfile(
                    name="title",
                    present_count=38,
                    total_count=38,
                    observed_types=["String"],
                ),
                "released": PropertyProfile(
                    name="released",
                    present_count=38,
                    total_count=38,
                    observed_types=["Long"],
                ),
            },
        ),
    },
    rel_type_profiles={
        "ACTED_IN": RelationshipTypeProfile(
            rel_type="ACTED_IN",
            count=253,
            source_labels={"Person"},
            target_labels={"Movie"},
            property_profiles={
                "role": PropertyProfile(
                    name="role",
                    present_count=253,
                    total_count=253,
                    observed_types=["String"],
                ),
            },
            cardinality_stats=CardinalityStats(count=120, min=1, max=8, mean=2.1),
        ),
    },
)

result_ok = compare_profile_to_definition(valid_profile, graph_definition)
print(
    f"Valid profile  — is_valid: {result_ok.is_valid}, issues: {len(result_ok.issues)}"
)

Valid profile  — is_valid: False, issues: 6


## 3. Build a profile with validation issues

Deliberate problems:
- `released` property missing from Movie (required → `MISSING_PROPERTY`).
- ACTED_IN `role` property has type `Long` instead of `String` (`PROPERTY_TYPE_MISMATCH`).
- ACTED_IN cardinality `min=0` violates `1..*` (`CARDINALITY_VIOLATION`).
- Unexpected `Genre` node label in profile (`UNEXPECTED_NODE_LABEL`).

In [4]:
invalid_profile = GraphProfile(
    source="neo4j://staging:7687",
    node_type_profiles={
        "Person": NodeTypeProfile(
            label="Person",
            count=50,
            property_profiles={
                "name": PropertyProfile(
                    name="name",
                    present_count=50,
                    total_count=50,
                    observed_types=["String"],
                ),
            },
        ),
        "Movie": NodeTypeProfile(
            label="Movie",
            count=20,
            property_profiles={
                "title": PropertyProfile(
                    name="title",
                    present_count=20,
                    total_count=20,
                    observed_types=["String"],
                ),
                # `released` is absent → MISSING_PROPERTY (ERROR)
            },
        ),
        "Genre": NodeTypeProfile(label="Genre", count=5),  # UNEXPECTED_NODE_LABEL
    },
    rel_type_profiles={
        "ACTED_IN": RelationshipTypeProfile(
            rel_type="ACTED_IN",
            count=40,
            source_labels={"Person"},
            target_labels={"Movie"},
            property_profiles={
                "role": PropertyProfile(
                    name="role",
                    present_count=40,
                    total_count=40,
                    observed_types=["Long"],  # wrong type → PROPERTY_TYPE_MISMATCH
                ),
            },
            cardinality_stats=CardinalityStats(
                count=50, min=0, max=5, mean=0.8
            ),  # min=0 violates 1..* → CARDINALITY_VIOLATION
        ),
    },
)

result_bad = compare_profile_to_definition(invalid_profile, graph_definition)
print(
    f"Invalid profile — is_valid: {result_bad.is_valid}, issues: {len(result_bad.issues)}"
)
print(f"                  errors:   {len(result_bad.errors)}")

Invalid profile — is_valid: False, issues: 8
                  errors:   4


## 4. Render all issues

In [5]:
for issue in result_bad.issues:
    print(f"[{issue.severity.value.upper():7}] {issue.code:<30} {issue.entity_id}")
    print(f"          {issue.message}")
    if issue.context:
        print(f"          context: {issue.context}")
    print()

[WARNING] UNEXPECTED_NODE_LABEL          Genre
          Profile contains node label 'Genre' not defined in model

[ERROR  ] MISSING_NODE_LABEL             City
          Model defines node type 'City' but no instances found in profile

[ERROR  ] MISSING_REL_TYPE               DIRECTED
          Model defines relationship type 'DIRECTED' but no instances found in profile

[ERROR  ] MISSING_REL_TYPE               LIVES_IN
          Model defines relationship type 'LIVES_IN' but no instances found in profile

[INFO   ] CONSTRAINT_UNVERIFIABLE        Movie.title
          Property 'title' on Movie is declared required but constraint information is unavailable for this backend/strategy (ADR-033)

[INFO   ] CONSTRAINT_UNVERIFIABLE        Person.name
          Property 'name' on Person is declared required but constraint information is unavailable for this backend/strategy (ADR-033)

[ERROR  ] PROPERTY_TYPE_MISMATCH         ACTED_IN.role
          Property 'role' on ACTED_IN has observed typ

## 5. Raise on errors (optional)

`ValidationResult.raise_on_errors()` converts collected errors into a
`GraphValidationError` exception — convenient for CI pipelines.

In [6]:
try:
    result_bad.raise_on_errors()
except GraphValidationError as exc:
    print("Caught GraphValidationError:")
    for issue in exc.issues:
        print(f"  [{issue.severity.value.upper()}] {issue.code}: {issue.message}")

Caught GraphValidationError:
  [ERROR] MISSING_NODE_LABEL: Model defines node type 'City' but no instances found in profile
  [ERROR] MISSING_REL_TYPE: Model defines relationship type 'DIRECTED' but no instances found in profile
  [ERROR] MISSING_REL_TYPE: Model defines relationship type 'LIVES_IN' but no instances found in profile
  [ERROR] PROPERTY_TYPE_MISMATCH: Property 'role' on ACTED_IN has observed type 'Long' (Python: int), expected str


## 6. All three comparison functions at a glance

In [7]:
from orthograph.comparison.engine import (
    compare_profile_to_definition,
)


print("compare_profile_to_definition — satisfaction check")
print("  Question: does the observed profile satisfy the declared constraints?")
print("  Default rules: standard_rules() (ERROR / WARNING / INFO)")
print("  is_valid = True only when zero ERRORs")
print()
print("compare_profiles — symmetric diff of two profiles")
print("  Question: what structural differences exist between two observed snapshots?")
print("  Default rules: diff_rules() (INFO only)")
print("  is_valid = always True")
print()
print("compare_definitions — symmetric diff of two definitions")
print("  Question: what changed between two declared schemas?")
print("  Default rules: diff_rules() (INFO only)")
print("  is_valid = always True")

compare_profile_to_definition — satisfaction check
  Question: does the observed profile satisfy the declared constraints?
  Default rules: standard_rules() (ERROR / WARNING / INFO)
  is_valid = True only when zero ERRORs

compare_profiles — symmetric diff of two profiles
  Question: what structural differences exist between two observed snapshots?
  Default rules: diff_rules() (INFO only)
  is_valid = always True

compare_definitions — symmetric diff of two definitions
  Question: what changed between two declared schemas?
  Default rules: diff_rules() (INFO only)
  is_valid = always True
